In [ ]:
# DFU Phase-4 Kaggle GPU V3 FIXED — supports Kaggle CLI zipped private input
import urllib.request, hashlib, shutil, zipfile
from pathlib import Path

VERSION = "DFU_PHASE4_KAGGLE_GPU_V3_FIXED_20260812"
SOURCE_COMMIT = "518c25a4dc3f103fad4a8e0545c3f6dd729c0434"
PARTS = [
    ("scripts/phase4_universal_v2_parts/part_01.pyfrag", "1280e699f2c02e74097cff69dd52db37cea77251"),
    ("scripts/phase4_universal_v2_parts/part_02.pyfrag", "ea776ea1476a6cbe41913f116887d605310c985f"),
    ("scripts/phase4_universal_v2_parts/part_03.pyfrag", "915202b01279b434fcbca6955c48b10bae13c942"),
    ("scripts/phase4_universal_v2_parts/part_04.pyfrag", "5e5128456b99877957dcecf341d3c149137acfbc"),
    ("scripts/phase4_universal_v2_parts/part_05.pyfrag", "db01f06aaa6519fef856903c20ba0f939ef5e14d"),
]
BASE = f"https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/{SOURCE_COMMIT}/"

def git_blob_sha(raw):
    return hashlib.sha1(b"blob " + str(len(raw)).encode() + b"\0" + raw).hexdigest()

print("=" * 100)
print(VERSION)
print("KAGGLE ZIP-INPUT FIX | GPU-REQUIRED INFERENCE | RESUME-SAFE")
print("NO TRAINING | NO FINE-TUNING | NO EXTERNAL THRESHOLD/CALIBRATION FITTING")
print("=" * 100)

chunks = []
for path, expected in PARTS:
    raw = urllib.request.urlopen(BASE + path, timeout=120).read()
    actual = git_blob_sha(raw)
    if actual != expected:
        raise RuntimeError(
            f"Source fragment mismatch for {path}: expected={expected} actual={actual}"
        )
    print("Source fragment PASS:", path, actual)
    chunks.append(raw)

source = b"".join(chunks)
text = source.decode("utf-8")
compile(text, "dfu_phase4_universal_v2.py", "exec")
print("Combined source compile: PASS")
print("Combined source SHA256:", hashlib.sha256(source).hexdigest())

# Load definitions without auto-running run()
ns = {"__name__": "dfu_phase4_universal_v2_module"}
exec(compile(text, "dfu_phase4_universal_v2.py", "exec"), ns)
_original_find_bundle = ns["find_bundle_root_under"]

def find_bundle_root_under_kaggle_fixed(search_root):
    search_root = Path(search_root)
    if not search_root.exists():
        return None

    manifests = sorted(search_root.rglob("FROZEN_INPUT_MANIFEST.json"))

    # Case A: already-unpacked bundle.
    for manifest in manifests:
        root = manifest.parent
        if (
            (root / "tables" / "14_checkpoint_inventory.csv").exists()
            and (root / "tables" / "all_oof_predictions.csv").exists()
            and (root / "tables" / "fold_seed_metrics.csv").exists()
            and (root / "checkpoints").exists()
        ):
            print("Frozen bundle found unpacked:", root)
            return root

    # Case B: Kaggle CLI '-r zip' upload: checkpoints.zip + tables.zip + top-level JSONs.
    for manifest in manifests:
        dataset_root = manifest.parent
        tables_zip = dataset_root / "tables.zip"
        checkpoints_zip = dataset_root / "checkpoints.zip"
        if not (tables_zip.exists() and checkpoints_zip.exists()):
            continue

        extract_root = Path("/kaggle/working/dfu_phase4_frozen_extracted")
        required_tables = [
            "14_checkpoint_inventory.csv",
            "all_oof_predictions.csv",
            "fold_seed_metrics.csv",
        ]

        def extracted_is_valid():
            return (
                (extract_root / "FROZEN_INPUT_MANIFEST.json").exists()
                and all((extract_root / "tables" / x).exists() for x in required_tables)
                and (extract_root / "checkpoints").exists()
            )

        if not extracted_is_valid():
            print("Detected Kaggle zipped frozen input. Extracting once...")
            if extract_root.exists():
                shutil.rmtree(extract_root)
            extract_root.mkdir(parents=True, exist_ok=True)

            for name in [
                "FROZEN_INPUT_MANIFEST.json",
                "PHASE2_FULL_VERIFICATION.json",
                "REPAIR45_FINAL_VERIFICATION.json",
            ]:
                src = dataset_root / name
                if src.exists():
                    shutil.copy2(src, extract_root / name)

            raw_tables = extract_root / "_tables_raw"
            raw_ckpt = extract_root / "_checkpoints_raw"
            raw_tables.mkdir(parents=True, exist_ok=True)
            raw_ckpt.mkdir(parents=True, exist_ok=True)

            with zipfile.ZipFile(tables_zip, "r") as zf:
                zf.extractall(raw_tables)
            with zipfile.ZipFile(checkpoints_zip, "r") as zf:
                zf.extractall(raw_ckpt)

            (extract_root / "tables").mkdir(parents=True, exist_ok=True)
            for name in required_tables:
                matches = [p for p in raw_tables.rglob(name) if p.is_file()]
                if len(matches) != 1:
                    raise RuntimeError(
                        f"Kaggle frozen input extraction failed for {name}: found {len(matches)} copies"
                    )
                shutil.copy2(matches[0], extract_root / "tables" / name)

            pd = ns["pd"]
            inv = pd.read_csv(extract_root / "tables" / "14_checkpoint_inventory.csv")
            if "bundle_relative_path" not in inv.columns:
                raise RuntimeError("Frozen checkpoint inventory lacks bundle_relative_path")

            for rel in inv["bundle_relative_path"].astype(str):
                dest = extract_root / rel
                basename = Path(rel).name
                matches = [p for p in raw_ckpt.rglob(basename) if p.is_file()]
                if len(matches) != 1:
                    raise RuntimeError(
                        f"Kaggle frozen checkpoint extraction failed for {basename}: "
                        f"found {len(matches)} copies"
                    )
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(matches[0], dest)

            shutil.rmtree(raw_tables, ignore_errors=True)
            shutil.rmtree(raw_ckpt, ignore_errors=True)

        if not extracted_is_valid():
            raise RuntimeError("Kaggle frozen bundle extraction did not produce the required layout")

        print("Kaggle frozen bundle extraction: PASS")
        print("Resolved frozen bundle:", extract_root)
        return extract_root

    # Preserve original behavior for other layouts / master ZIP.
    return _original_find_bundle(search_root)

ns["find_bundle_root_under"] = find_bundle_root_under_kaggle_fixed

print("Kaggle private-input compatibility patch: PASS")
print("Starting Phase-4...")
ns["run"]()
